In [6]:
import h5py

f = h5py.File("C:/Users/lucy7/OneDrive/Documents/2026/HONOURS/3D_Simulations/Project_RRT/human_demos/1788835639_254104/image.hdf5", "r")

print("num demos:", len(f["data"].keys()))
print("total attr:", f["data"].attrs.get("total", "none"))

demo_keys = list(f["data"].keys())
if len(demo_keys) > 0:
    d0 = f["data"][demo_keys[0]]
    print(f"\n{demo_keys[0]} keys:", list(d0.keys()))
    
    if "obs" in d0:
        print("obs sub-keys:", list(d0["obs"].keys()))
        for k in d0["obs"].keys():
            print(f"  {k}: shape={d0['obs'][k].shape}, dtype={d0['obs'][k].dtype}")
    else:
        print("NO 'obs' GROUP FOUND — conversion did not populate observations")
    
    if "next_obs" in d0:
        print("next_obs present, sub-keys:", list(d0["next_obs"].keys()))
    
    print("actions shape:", d0["actions"].shape)
    print("rewards shape:", d0["rewards"].shape if "rewards" in d0 else "no rewards")
    print("dones shape:", d0["dones"].shape if "dones" in d0 else "no dones")
else:
    print("STILL EMPTY — zero demos")

f.close()

num demos: 200
total attr: 48308

demo_1 keys: ['action_dict', 'actions', 'dones', 'next_obs', 'obs', 'rewards', 'states']
obs sub-keys: ['agentview_image', 'object', 'robot0_eef_pos', 'robot0_eef_quat', 'robot0_eef_quat_site', 'robot0_eye_in_hand_image', 'robot0_gripper_qpos', 'robot0_gripper_qvel', 'robot0_joint_acc', 'robot0_joint_pos', 'robot0_joint_pos_cos', 'robot0_joint_pos_sin', 'robot0_joint_vel']
  agentview_image: shape=(252, 84, 84, 3), dtype=uint8
  object: shape=(252, 12), dtype=float64
  robot0_eef_pos: shape=(252, 3), dtype=float64
  robot0_eef_quat: shape=(252, 4), dtype=float64
  robot0_eef_quat_site: shape=(252, 4), dtype=float32
  robot0_eye_in_hand_image: shape=(252, 84, 84, 3), dtype=uint8
  robot0_gripper_qpos: shape=(252, 2), dtype=float64
  robot0_gripper_qvel: shape=(252, 2), dtype=float64
  robot0_joint_acc: shape=(252, 7), dtype=float64
  robot0_joint_pos: shape=(252, 7), dtype=float64
  robot0_joint_pos_cos: shape=(252, 7), dtype=float64
  robot0_joint_pos_

In [8]:
import numpy as np

f = h5py.File("C:/Users/lucy7/OneDrive/Documents/2026/HONOURS/3D_Simulations/Project_RRT/human_demos/1788835639_254104/image.hdf5", "r")
d0 = f["data"][list(f["data"].keys())[0]]

for k in d0["obs"].keys():
    arr = d0["obs"][k][:]
    print(f"{k}: min={arr.min():.4f}, max={arr.max():.4f}, mean={arr.mean():.4f}")

f.close()

agentview_image: min=1.0000, max=255.0000, mean=213.2166
object: min=-0.1257, max=1.0000, mean=0.3073
robot0_eef_pos: min=-0.1257, max=1.0037, mean=0.2950
robot0_eef_quat: min=-0.0280, max=0.9986, mean=0.2602
robot0_eef_quat_site: min=0.0323, max=0.7258, mean=0.3736
robot0_eye_in_hand_image: min=3.0000, max=254.0000, mean=195.9263
robot0_gripper_qpos: min=-0.0400, max=0.0400, mean=0.0006
robot0_gripper_qvel: min=-0.0635, max=0.0665, mean=0.0001
robot0_joint_acc: min=-2.4543, max=10.6893, mean=0.9238
robot0_joint_pos: min=-2.6670, max=3.0909, mean=0.3058
robot0_joint_pos_cos: min=-0.9987, max=1.0000, mean=0.4078
robot0_joint_pos_sin: min=-0.8723, max=0.7778, mean=0.0967
robot0_joint_vel: min=-0.7104, max=0.6543, mean=0.0057


In [1]:
import huggingface_hub
print(huggingface_hub.__version__)
print(huggingface_hub.__file__)

0.23.4
/home/ubuntu/.venv/lib/python3.12/site-packages/huggingface_hub/__init__.py


In [2]:
import os
import diffusers
from diffusers.training_utils import EMAModel
from robomimic.algo.diffusion_policy import DiffusionPolicyUNet
print("diffusers:", diffusers.__version__)
print("imports OK")

ModuleNotFoundError: No module named 'requests'

In [ ]:
from environments.LIFT_withRandFriction import LiftCube_RandGap # necessary if enable in-the-loop rollout

In [4]:
import json
from robomimic.config import config_factory

# start from the DP template rather than hand-writing one
#with open("robomimic/robomimic/exps/templates/diffusion_policy.json", "r") as f:
with open("C:/Users/lucy7/miniconda3/envs/robosuite/lib/site-packages/robomimic/exps/templates/diffusion_policy.json", "r") as f: #because of the cd above
    ext_cfg = json.load(f)

config = config_factory(ext_cfg["algo_name"])   # "diffusion_policy"
with config.values_unlocked():
    config.update(ext_cfg)

In [ ]:
import time

with config.values_unlocked():
    # core DP receding-horizon structure
    config.algo.horizon.observation_horizon = 2
    config.algo.horizon.action_horizon = 8
    config.algo.horizon.prediction_horizon = 16

    # noise scheduler -- ddpm (slower, higher quality) vs ddim (fast inference)
    config.algo.ddpm.enabled = True
    config.algo.ddim.enabled = False

    # dataset path and training length
    config.train.data = "C:/Users/lucy7/OneDrive/Documents/2026/HONOURS/3D_Simulations/Project_RRT/human_demos/1788835639_254104/image.hdf5"   # or low_dim.hdf5
    config.train.num_epochs = 500 #start with less epochs for test runs. 
    config.train.batch_size = 64 #used in the paper

    # Disable in-the-loop rollout
    config.experiment.rollout.enabled = False

    #name of this run
    config.experiment.name = f"test_{int(time.time())}"

    '''
    #State-Based Observation Config
    config.observation.modalities.obs.low_dim = [
        "robot0_eef_pos", "robot0_eef_quat", "robot0_gripper_qpos",
        "object",
    ]
    config.observation.modalities.obs.rgb = []
    '''

    #Vision-Based Observation Config
    config.observation.modalities.obs.low_dim = [
        "robot0_eef_pos", "robot0_eef_quat", "robot0_gripper_qpos",
    ]
    config.observation.modalities.obs.rgb = ["agentview_image", "robot0_eye_in_hand_image"]

    config.observation.encoder.rgb.core_class = "VisualCore"  # already the template default

    # NOT in the template by default (it's null) -- worth adding given only 100 demos,
    # since a vision policy will overfit fast without augmentation
    config.observation.encoder.rgb.obs_randomizer_class = "CropRandomizer"
    config.observation.encoder.rgb.obs_randomizer_kwargs = {
        "crop_height": 76, "crop_width": 76, "num_crops": 1, "pos_enc": False,
    }



In [ ]:
import torch
from robomimic.scripts.train import train

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
train(config, device=device)